<a href="https://colab.research.google.com/github/mayaannkkk/LangChain-V1/blob/main/RAG_USING_LANGCHAIN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain-community FAISS-cpu tiktoken python-dotenv

In [ ]:
!pip install youtube-transcript-api

# Indexing - text loader

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi
video_id = "_3ezSpJw2E8"

ytt_api = YouTubeTranscriptApi()
trascript_list = ytt_api.fetch(video_id, languages=["hi"])

transcript = " ".join(snippet.text for snippet in trascript_list)
print(transcript)

# Indexing = text splitting

In [ ]:
!pip install RecursiveCharacterTextSplitter

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [ ]:
chunks[0]

In [ ]:
import os
from getpass import getpass

os.environ["HF_TOKEN"] = getpass("Enter Hugging Face token: ")

In [ ]:
!pip install langchain-huggingface

# Embedding

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# Vector store

In [ ]:
from langchain_community.vectorstores import FAISS
vector_store = FAISS.from_documents(chunks,embeddings)

# Retrieval

In [ ]:
retriever = vector_store.as_retriever(search_type = "similarity", search_kwargs = {"k":4})

In [ ]:
retriever

In [ ]:
retriever.invoke("WHat is langchain")

# AUgmentation

In [ ]:
from langchain_core.prompts import PromptTemplate
prompt = PromptTemplate(
    template="""You are a helpful chatassistant
       Give response only from the provided text
       If you don't know the answer just say no

       {context}
       Question: {question}
       Answer:""",
    input_variables=["context", "question"],
)

In [ ]:
question          = "is the topic of lagchain discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [ ]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

In [ ]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [ ]:
import os
from getpass import getpass

os.environ["HF_TOKEN"] = getpass("Enter your Hugging Face token: ")

In [ ]:
!pip install langchain-groq


In [ ]:
from langchain_groq import ChatGroq

In [ ]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter your GROQ_API_KEY: ")

In [ ]:
llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    api_key = os.environ.get("GROQ_API_KEY")
)

In [ ]:

answer = llm.invoke(final_prompt)
print(answer.content)

# chain

In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda

In [ ]:
def format_doc(retrieved_docs):
  context = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context

In [ ]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_doc),
    'question': RunnablePassthrough()
})

In [ ]:
from langchain_core.output_parsers import StrOutputParser

In [ ]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
main_chain.invoke("langchain kya hai")